# CS781 Phase 7 - Prior Case Retrieval - notebook `5`

**Task.** Prior case retrieval: rank each query's candidate pool, output the top 7. Primary metric micro-F1@7.
**Lexical stage** (CPU, setup cell): BM25 over word 1-6-grams of the whole query (n-gram BM25: U-CREAT 2023), BM25 of
each citation passage on its own (best passage counts), reporter-citation match (a citation written in the query, e.g.
"2 SCR 148", that is the candidate's own), a date prior; weights tuned on TRAIN only (dev micro-F1@7 0.469 offline).
Its top-50 non-gold candidates are the hard negatives.
**Neural retriever** (each run): a Phase-1 model as a bi-encoder. The QUERY is its set of citation passages, each
encoded alone; a CANDIDATE is the mean of its 256-token chunk vectors; score = best passage (max). InfoNCE with in-batch
+ lexical hard negatives; 8% of train queries pick the checkpoint. dev/test are encoded in full; four dense scores
(passage-max vs document vector / vs best chunk, document cosine, chunk MaxSim) are each fused with the lexical score
(weight chosen on dev; a cross-fitted dev estimate is reported too).
**Knowledge distillation:** B01 (110M teacher) -> E (58M): KL to the teacher's score distribution over each batch;
E03 = same student without a teacher. **Z\* runs:** each Phase-1 model with no training (transfer baseline).

| GPU | run | technique | training budget | question |
|---|---|---|---|---|
| GPU0 | `B01_full` | Full FT | 150 min | full fine-tuning of the GPT model; also the distillation teacher |
| GPU0 | `E02_kd` | Knowledge distillation (B01 -> E) | 150 min | does the 58M student keep the 110M teacher's retrieval quality? |
| GPU1 | `E03_full` | Full FT (no teacher) | 150 min | the student without a teacher |
| GPU1 | `ZE_zeroshot` | none (Phase-1 model as is) | 0 min | how well do Phase-1 representations retrieve without any training? |
| GPU1 | `ZB_zeroshot` | none (Phase-1 model as is) | 0 min | how well do Phase-1 representations retrieve without any training? |

**Inputs:** dataset `cs781-p7-code`, `cs781-legal-data` (tokenizer), Phase-1 checkpoint(s) of model B + E.
Internet ON (Hugging Face data), secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
If the Phase-7 data is not on Hugging Face, attach a dataset with the six json files - they are found automatically.
Expected: ~7.0 h (setup ~20-30 min: tokenization + lexical stage; smoke test ~10 min).
**Outputs:** `submissions/X00_lexical.zip` (after setup), `submissions/<run>_fused.zip` and `<run>_dense.zip` (as soon as
each run finishes), `models/<run>.pt`, `bundle_p7_5.zip` (dev/test scores, logs), `REPORT_p7_5.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
import numpy as np
NB_START = time.time()
W = "/kaggle/working"
WORK, P7DATA, RUNS, MODELS = f"{W}/p7code", "/tmp/p7data", f"{W}/runs", f"{W}/models"
CACHE = "/tmp/p7cache"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions", P7DATA):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("pcr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-7 data: Hugging Face (Phase_7*/...), else any attached *.json with the expected names
import pcr_data
try:
    from huggingface_hub import hf_hub_download, list_repo_files
    from kaggle_secrets import UserSecretsClient
    REPO = "Exploration-Lab/CS-781-Capstone"
    tok_ = UserSecretsClient().get_secret("HF_TOKEN")
    files = [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_7")]
    print("HF files:", files)
    for f in files:
        dst = os.path.join(P7DATA, f); os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)
except Exception as ex:
    print("Hugging Face download failed:", repr(ex))
if len(pcr_data.find_files(P7DATA)) < 3:
    print("using attached files:", pcr_data.find_files("/kaggle/input"))
    for s, (q, c) in pcr_data.find_files("/kaggle/input").items():
        os.makedirs(f"{P7DATA}/{s}", exist_ok=True)
        shutil.copy(q, f"{P7DATA}/{s}/{s}.json"); shutil.copy(c, f"{P7DATA}/{s}/{s}_candidates.json")
print("data files:", pcr_data.find_files(P7DATA))

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
ALL_PT = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("checkpoint files found under /kaggle/input:", *ALL_PT, sep="\n  ")
for p in ALL_PT:
    if os.path.basename(p) != "final.pt" or "smoke" in p.lower():
        continue
    m = re.search(r"run[_-]([a-e])(?:[_/-]|$)", p, re.I) or re.search(r"model[_-]?([a-e])(?:[_/-]|$)", p, re.I)
    if not m:
        continue
    k = m.group(1).lower()
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
CKPT.update(CKPT_OVERRIDE if "CKPT_OVERRIDE" in globals() else {})   # e.g. CKPT_OVERRIDE = {"c": "/kaggle/input/.../final.pt"}
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- unit tests; cache: words, citation contexts, years, BPE ids (4 processes); lexical stage (BM25 n-grams)
r = subprocess.run(f"python tests_pcr.py {TOK_ROOT}", shell=True, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"
t0 = time.time()
STATS = pcr_data.build(P7DATA, TOK_ROOT, CACHE, workers=4)
print(json.dumps(STATS, indent=1)); print(f"cache built in {(time.time() - t0) / 60:.1f} min")
import pcr_lex
t0 = time.time()
LEX = pcr_lex.run(CACHE)
print(f"lexical stage in {(time.time() - t0) / 60:.1f} min")
for f in ("meta.json", "lex_report.json"):
    shutil.copy(f"{CACHE}/{f}", f"{W}/{f}")

# lexical-only submission (no neural model): a reference point and a safe early submission
_m = json.load(open(f"{CACHE}/meta.json"))["test"]
_lx = np.load(f"{CACHE}/lex.npz")
_pred = pcr_data.topk(_lx["test_lex_nocit"], _m["cids"], _m["qids"])
pcr_data.write_prediction(f"{W}/prediction.json", _m["qids"], _pred, _m["cids"])
with zipfile.ZipFile(f"{W}/submissions/X00_lexical.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{W}/prediction.json", arcname="prediction.json")
np.savez_compressed(f"{W}/lex_scores.npz", dev_lex=_lx["dev_lex"].astype(np.float32), test_lex=_lx["test_lex"].astype(np.float32),
                    dev_lex_nocit=_lx["dev_lex_nocit"].astype(np.float32), test_lex_nocit=_lx["test_lex_nocit"].astype(np.float32),
                    **{k: _lx[k] for k in _lx.files if k.startswith(("dev__", "test__"))})
print("written submissions/X00_lexical.zip (no CITATOR link); dev micro-F1@7:", LEX["lex_nocit_f1"], "| with CITATOR:", LEX["lex_f1"])

code: ['attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'p1_loader.py', 'pcr_data.py', 'pcr_lex.py', 'pcr_model.py', 'pcr_train.py', 'peft.py', 'tests_pcr.py']
HF files: ['Phase_7_PCR/Dev/dev.json', 'Phase_7_PCR/Dev/dev_candidates.json', 'Phase_7_PCR/Test/test.json', 'Phase_7_PCR/Test/test_candidates.json', 'Phase_7_PCR/Train/train.json', 'Phase_7_PCR/Train/train_candidates.json']


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

... (2 outputs left out)

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

data files: {'train': ('/tmp/p7data/Phase_7_PCR/Train/train.json', '/tmp/p7data/Phase_7_PCR/Train/train_candidates.json'), 'dev': ('/tmp/p7data/Phase_7_PCR/Dev/dev.json', '/tmp/p7data/Phase_7_PCR/Dev/dev_candidates.json'), 'test': ('/tmp/p7data/Phase_7_PCR/Test/test.json', '/tmp/p7data/Phase_7_PCR/Test/test_candidates.json')}
checkpoint files found under /kaggle/input:
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/latest.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/py

In [2]:
NB_HOURS = 11.0                                   # Kaggle stops at 12 h; no run starts after 11 h and runs are cut to fit
INF_MIN = 40                                      # time kept for encoding the dev + test pools after training

def cmd(e, out, extra, model_dir):
    args = e["args"].replace("{MODEL_DIR}/", model_dir + ("/smoke_" if smoke_mode[0] else "/"))
    args = re.sub(r"\{CKPT:(\w)\}", lambda m: CKPT[m.group(1)], args)
    return (f"python pcr_train.py --ckpt {CKPT[e['model']]} --cache {CACHE} --out {out} --model_dir {model_dir} "
            f"{args} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               TOKENIZERS_PARALLELISM="false")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-400:], flush=True)
    if not smoke_mode[0]:
        for kind in ("fused", "dense"):
            f = f"{RUNS}/{tag}/submission_{kind}.zip"
            if os.path.exists(f):
                shutil.copy(f, f"{W}/submissions/{tag}_{kind}.zip")     # ready as soon as the run ends
    return p.returncode

smoke_mode = [False]
SMOKE = "--smoke --steps 2 --batch 2 --hard 1 --q_chunks 1 --q_ctx 2 --d_chunks 1 --eval_every 1000 --inf_tok 512 --inf_batch 8"

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel."""
    smoke_mode[0] = smoke
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" + e["tag"]) if smoke else e["tag"]
            if smoke:
                extra = SMOKE.replace("--steps 2", "--steps 0") if "--steps 0" in e["args"] else SMOKE
            else:
                left = NB_HOURS * 60 - (time.time() - NB_START) / 60
                if left < INF_MIN + 15:
                    print(f"[GPU{gpu}] {t}: SKIPPED (notebook time budget used up)", flush=True)
                    status[t] = "skipped"
                    continue
                extra = f"--max_minutes {max(10, min(e['train_min'], left - INF_MIN - 10)):.0f}"
            try:
                status[t] = run(t, cmd(e, f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
            except Exception as ex:
                print(f"[GPU{gpu}] {t}: NOT STARTED - {ex!r}", flush=True)
                open(f"{W}/logs/{t}.txt", "w").write(f"not started: {ex!r}")
                status[t] = "error"
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Every experiment on a few queries, 2 steps, short documents. A failing experiment is dropped (its error is
    printed) instead of stopping the notebook."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    keep = [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]
    tags = {e["tag"] for q in keep for e in q}            # a distillation run needs its teacher
    return [[e for e in q if "{MODEL_DIR}/" not in e["args"] or
             re.search(r"\{MODEL_DIR\}/(\S+)\.pt", e["args"]).group(1) in tags] for q in keep]

## Experiments (one queue per GPU)

In [3]:
NAME = '5'
QUEUES = [[{'tag': 'B01_full', 'model': 'b', 'tech': 'Full FT', 'args': '--method full --lr 3e-5', 'train_min': 150},
  {'tag': 'E02_kd',
   'model': 'e',
   'tech': 'Knowledge distillation (B01 -> E)',
   'args': '--method full --lr 5e-5 --teacher {MODEL_DIR}/B01_full.pt --teacher_ckpt {CKPT:b} --kd_alpha 0.5 --kd_T 1',
   'train_min': 150}],
 [{'tag': 'E03_full', 'model': 'e', 'tech': 'Full FT (no teacher)', 'args': '--method full --lr 5e-5', 'train_min': 150},
  {'tag': 'ZE_zeroshot', 'model': 'e', 'tech': 'none (Phase-1 model as is)', 'args': '--method full --steps 0', 'train_min': 0},
  {'tag': 'ZB_zeroshot', 'model': 'b', 'tech': 'none (Phase-1 model as is)', 'args': '--method full --steps 0', 'train_min': 0}]]
ALL_QUEUES = [list(q) for q in QUEUES]

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_E03_full: exit 0 | DEV micro-F1@7: lexical 0.0000 | dense dense_mean 0.0000 | fused lex+0.05*mean 0.0000 (cross-fitted 0.0000) / done in 0.3 min
[GPU1] smoke_ZE_zeroshot: exit 0 | DEV micro-F1@7: lexical 0.0000 | dense dense_mean 0.0000 | fused lex+0.05*mean 0.0000 (cross-fitted 0.0000) / done in 0.1 min
[GPU0] smoke_B01_full: exit 0 | DEV micro-F1@7: lexical 0.0000 | dense dense_mean 0.0000 | fused lex+0.05*mean 0.0000 (cross-fitted 0.0000) / done in 0.5 min
[GPU1] smoke_ZB_zeroshot: exit 0 | DEV micro-F1@7: lexical 0.0000 | dense dense_mean 0.0000 | fused lex+0.05*mean 0.0000 (cross-fitted 0.0000) / done in 0.1 min
[GPU0] smoke_E02_kd: exit 0 | DEV micro-F1@7: lexical 0.0000 | dense dense_mean 0.0000 | fused lex+0.05*mean 0.0000 (cross-fitted 0.0000) / done in 0.3 min
smoke done in 0.8 min: {'smoke_E03_full': 0, 'smoke_ZE_zeroshot': 0, 'smoke_B01_full': 0, 'smoke_ZB_zeroshot': 0, 'smoke_E02_kd': 0}
SMOKE OK


## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU1] E03_full: exit 0 | DEV micro-F1@7: lexical 0.4691 | dense dense_ctxdoc 0.3624 | fused lex+0.1*maxsim 0.4814 (cross-fitted 0.4736) / done in 91.4 min
[GPU1] ZE_zeroshot: exit 0 | DEV micro-F1@7: lexical 0.4691 | dense dense_maxsim 0.2991 | fused lex+0.05*mean 0.4780 (cross-fitted 0.4725) / done in 4.9 min
[GPU1] ZB_zeroshot: exit 0 | DEV micro-F1@7: lexical 0.4691 | dense dense_ctxchunk 0.2991 | fused lex+0.05*mean 0.4769 (cross-fitted 0.4725) / done in 8.0 min
[GPU0] B01_full: exit 0 | DEV micro-F1@7: lexical 0.4691 | dense dense_ctxdoc 0.3747 | fused lex+0.05*mean 0.4803 (cross-fitted 0.4747) / done in 145.6 min
[GPU0] E02_kd: exit 0 | DEV micro-F1@7: lexical 0.4691 | dense dense_ctxdoc 0.3658 | fused lex+0.1*maxsim 0.4814 (cross-fitted 0.4792) / done in 126.4 min
runs done in 272.1 min: {'E03_full': 0, 'ZE_zeroshot': 0, 'ZB_zeroshot': 0, 'B01_full': 0, 'E02_kd': 0}


## 3. Report

In [6]:
def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT p7 {name} ===", "lexical stage: " + json.dumps({k: LEX[k] for k in ("weights", "fused", "dev_prf", "dev_recall@")}),
         "run                  | technique                       | dev dense | choice       | dev fused | choice            | crossfit | val@best | best step | trainable | train min | tok/s  | train GB | enc s/doc"]
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            lg = f"{W}/logs/{e['tag']}.txt"
            tail = " / ".join(open(lg).read().strip().splitlines()[-3:])[-300:] if os.path.exists(lg) else "not run"
            L.append(f"{e['tag']:20s} | FAILED/SKIPPED: {tail}")
            continue
        s = json.load(open(p))
        L.append(f"{e['tag']:20s} | {e['tech'][:31]:31s} | {s['dev_dense']:.4f}    | {s['dense_choice']:12s} | "
                 f"{s['dev_fused']:.4f}    | {s['fused_choice']:17s} | {s['dev_fused_crossfit']:.4f}   | {s['best_val']:.4f}   | "
                 f"{s['best_step']:9d} | {s['trainable'] / 1e6:8.2f}M | {s['train_minutes']:9.0f} | {s['train_tok_s']:6,d} | "
                 f"{s['peak_train_gb']:8.1f} | {s['encode_s_per_doc']:.3f}")
        L.append(f"{'':20s}   dev P/R@7 dense {s['dev'][s['dense_choice']]['p']:.4f}/{s['dev'][s['dense_choice']]['r']:.4f}, "
                 f"fused {s['dev'][s['fused_choice']]['p']:.4f}/{s['dev'][s['fused_choice']]['r']:.4f}; val history " +
                 " ".join(f"{h['step']}:{h['val']:.3f}" for h in s["history"]))
    bundle = f"{W}/bundle_p7_{name}.zip"              # everything except model weights
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for f in ("meta.json", "lex_report.json", "lex_scores.npz"):
            z.write(f"{W}/{f}", arcname=f)
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for root, _, fs in os.walk(d):
                    for f in fs:
                        z.write(os.path.join(root, f), arcname=os.path.join(e["tag"], os.path.relpath(os.path.join(root, f), d)))
            if os.path.exists(f"{W}/logs/{e['tag']}.txt"):
                z.write(f"{W}/logs/{e['tag']}.txt", arcname=f"{e['tag']}/stdout.txt")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_p7_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT p7 5 ===
lexical stage: {"weights": {"ctxmax_4": 0.5, "later": -8, "bm25_3": 0, "rep": 4, "citator": 4, "bm25_5": 0, "bm25_6": 0.5}, "fused": {"train": 0.4246, "dev": 0.4714}, "dev_prf": {"f1": 0.4714, "p": 0.5133, "r": 0.4358, "tp": 424}, "dev_recall@": {"7": 0.4358, "20": 0.6249, "50": 0.7153, "100": 0.7657, "200": 0.8078}}
run                  | technique                       | dev dense | choice       | dev fused | choice            | crossfit | val@best | best step | trainable | train min | tok/s  | train GB | enc s/doc
B01_full             | Full FT                         | 0.3747    | dense_ctxdoc | 0.4803    | lex+0.05*mean     | 0.4747   | 0.3458   |      1500 |   110.12M |       137 |  9,225 |      6.4 | 0.149
                       dev P/R@7 dense 0.4080/0.3464, fused 0.5230/0.4440; val history 0:0.281 250:0.307 500:0.320 750:0.323 1000:0.330 1250:0.323 1500:0.346 1750:0.343 2000:0.343
E02_kd               | Knowledge distillation (B01 ->  | 0.3658    | dense_ct